In [1019]:
import torch

In [1020]:
def sample_sphere(n):
    """Sample a point on a sphere"""
    z = 2*torch.rand(n) - 1
    phi = 2*(torch.pi)*torch.rand(n)
    r = torch.sqrt(1-z**2)
    x, y = r*torch.cos(phi), r*torch.sin(phi)
    return torch.concat([x.reshape(-1,1),y.reshape(-1,1),z.reshape(-1,1)], axis=1)
    

In [1021]:
coord = sample_sphere(4)

In [1022]:
coord

tensor([[-0.3940,  0.7045, -0.5903],
        [-0.7531,  0.3764, -0.5396],
        [-0.9853, -0.0257, -0.1688],
        [ 0.4081,  0.9099,  0.0742]])

In [1023]:
torch.square(coord)

tensor([[1.5520e-01, 4.9637e-01, 3.4844e-01],
        [5.6718e-01, 1.4167e-01, 2.9114e-01],
        [9.7086e-01, 6.6190e-04, 2.8478e-02],
        [1.6658e-01, 8.2792e-01, 5.5038e-03]])

In [1024]:
def check_validity(coord):
    assert  torch.all(torch.abs(torch.sum((torch.square(coord)), axis=1)  - 1) <= 1e-3)

In [1025]:
check_validity(coord)

In [1026]:
x,y = sample_sphere(4), sample_sphere(4)

In [1027]:
x.shape, y.shape

(torch.Size([4, 3]), torch.Size([4, 3]))

In [1028]:

def log_map(x,y):
    """Finds the minimum geodesic or minor arc for circle"""
    assert x.shape==y.shape
    n = x.shape[0]
    dot_prod = torch.tensor([torch.dot(x[i],y[i]) for i in range(n)]).reshape(-1,1)
    theta = torch.tensor([torch.arccos(dot) for dot in dot_prod]).reshape(-1,1)
    y_parallel = dot_prod * x
    y_perp = y - y_parallel
    y_perp_norm = torch.nn.functional.normalize(y_perp, 2, dim=-1)
    v = theta * y_perp_norm
    return v


        

In [1029]:
v = log_map(x,y)

In [1030]:
v.shape

torch.Size([4, 3])

In [1031]:
def test_tangentness(x, v, tol=1e-5):
    assert x.shape==v.shape
    n = x.shape[0]
    dot_prod = torch.tensor([torch.dot(x[i],v[i]) for i in range(n)])
    tol_check = torch.abs(dot_prod)<=tol
    assert torch.all(tol_check)


In [1032]:
test_tangentness(x,v)

In [1033]:
def log_map_length_test(x,y,v, tol=1e-5):
    assert x.shape == y.shape
    assert x.shape == v.shape
    n = x.shape[0]
    theta = torch.tensor([torch.arccos(torch.dot(x[i], y[i])) for i in range(n)])
    v_norm = torch.linalg.vector_norm(v, ord=2, dim=-1)
    assert torch.all(torch.abs(v_norm - theta) <= tol)


In [1034]:
log_map_length_test(x,y,v)

In [1035]:
def exp_map(x,v):
    assert x.shape==v.shape
    n = x.shape[0]
    v_norm = torch.nn.functional.normalize(v, p=2, dim=-1)
    theta = torch.linalg.vector_norm(v, ord=2, dim=-1)
    y = torch.cos(theta).reshape(-1,1) * x + torch.sin(theta).reshape(-1,1) * v_norm
    return y
    

In [1036]:
exp_map(x,v)

tensor([[ 0.0108,  0.8166,  0.5771],
        [-0.8037,  0.1573, -0.5738],
        [ 0.8478, -0.2886,  0.4449],
        [-0.0538,  0.9326,  0.3567]])

In [1037]:
def test_exp(x,y, tol=1e-5):
    assert torch.all(torch.abs(exp_map(x, log_map(x,y)) - y) <= tol)

In [1038]:
test_exp(x,y)

In [1039]:
def test_unit_norm(x,v, tol=1e-5):
    assert torch.all(torch.abs(torch.linalg.vector_norm(exp_map(x,v), ord=2, dim=-1)-1) <= tol)

In [1040]:
test_unit_norm(x,v)

In [1041]:
def premetric_d(x,y):
    assert x.shape == y.shape
    n = x.shape[0]
    return torch.tensor(
        [torch.arccos(torch.clamp(torch.dot(x[i], y[i]), min=-1, max=1)) for i in range(n)]
        )

In [1042]:
premetric_d(x,y)

tensor([1.3934, 1.7697, 1.4306, 1.5026])

In [1043]:
def test_d_x_x_zero(x,tol=1e-3):
    assert torch.all(
        torch.abs(
            premetric_d(x,x) - 0
        ) <= tol
    )

In [1044]:
test_d_x_x_zero(x)

In [1045]:
def test_d_symmetry(x,y,tol=1e-3):
    assert torch.all(
        torch.abs(
            premetric_d(x,y) - premetric_d(y,x)
        ) <= tol
    )

In [1046]:
test_d_symmetry(x,y)

In [1047]:
def test_d_non_neg(x,y):
    assert torch.all(
        premetric_d(x,y) >= 0
    )

In [1048]:
test_d_non_neg(x,y)

In [1049]:
def grad_d(x, y):
    v = log_map(x,y)
    v_norm = torch.linalg.vector_norm(v, ord=2, dim=-1).reshape(-1,1)
    return -v * torch.reciprocal(v_norm)

In [1050]:
g = grad_d(x,y)

In [1051]:
test_tangentness(x,g)

In [1052]:
test_unit_norm(x,g)

In [1053]:
def test_against_log_map(x,y, tol=1e-3):
    assert torch.all(torch.abs(
        torch.nn.functional.normalize(log_map(x,y),p=2,dim=-1) + grad_d(x,y)
    ) <= tol
    )

In [1054]:
test_against_log_map(x,y)

In [1055]:
def get_time_sched_and_derivative(t):
    assert torch.all(t < 1)
    assert torch.all(t>=0)
    return 1-t, -1*torch.reciprocal(1-t)

In [1056]:
get_time_sched_and_derivative(torch.tensor([0,0.5]))

(tensor([1.0000, 0.5000]), tensor([-1., -2.]))

In [1057]:
def conditional_vf(x, x1, t):
    assert x.shape == x1.shape
    assert x.shape[0] == t.shape[0]
    grad = grad_d(x,x1)
    pre = premetric_d(x,x1).reshape(-1,1)
    _, log_deriv = get_time_sched_and_derivative(t)
    log_deriv = log_deriv.reshape(-1,1)
    return log_deriv * pre * grad * \
            torch.reciprocal(
                torch.square(
                torch.linalg.vector_norm(grad,ord=2,dim=-1)
                ).reshape(-1,1)
                )

    


In [1058]:
def get_time_samples(n):
    return torch.rand(n)

In [1059]:
t = get_time_samples(4)

In [1060]:
u = conditional_vf(x,y,t)

In [1061]:
test_tangentness(x,u)

In [1062]:
t.shape

torch.Size([4])

In [1063]:
def geodesic_path(x0, x1, t):
    return exp_map(x0, t.reshape(-1,1)*log_map(x0,x1))

In [1064]:
geodesic_path(x,y,t)

tensor([[-0.9305,  0.3657,  0.0212],
        [-0.9089, -0.0495, -0.4141],
        [ 0.6417, -0.3401,  0.6874],
        [ 0.5531,  0.8274, -0.0976]])

In [1065]:
def test_start(x, y, tol=1e-3):
    assert x.shape == y.shape
    n = x.shape[0]
    t = torch.zeros(n,1)
    assert torch.all(torch.abs(geodesic_path(x,y,t) - x)<=tol)


In [1066]:
test_start(x,y)

In [1067]:
def test_end(x, y, tol=1e-3):
    assert x.shape == y.shape
    n = x.shape[0]
    t = torch.ones(n,1)
    assert torch.all(torch.abs(geodesic_path(x,y,t) - y)<=tol)

In [1068]:
test_end(x,y)

In [1069]:
def test_distance_schedule(x0, x1, t, tol=1e-3):
    t = t.reshape(-1, 1)
    xt = exp_map(x0, t * log_map(x0, x1))
    assert torch.all(
        torch.abs(
            premetric_d(xt, x1).reshape(-1,1) - ((1-t) * premetric_d(x0, x1).reshape(-1,1))
        ) <= tol
        
    )

In [1070]:
test_distance_schedule(x,y,t)

In [1071]:
t_sweep = torch.tensor([0, 0.25, 0.5, 0.75, 1])
n=4
t_sweep.repeat(n,1)

tensor([[0.0000, 0.2500, 0.5000, 0.7500, 1.0000],
        [0.0000, 0.2500, 0.5000, 0.7500, 1.0000],
        [0.0000, 0.2500, 0.5000, 0.7500, 1.0000],
        [0.0000, 0.2500, 0.5000, 0.7500, 1.0000]])

In [1078]:
def test_sweep(x0, x1, tol=1e-5):
    assert x0.shape == x1.shape
    n = x0.shape[0]
    t_sweep = torch.tensor([0, 0.25, 0.5, 0.75, 1]).repeat(n, 1)
    n_t = t_sweep.shape[1]
    for i in range(n_t):
        test_distance_schedule(x0, x1, t_sweep[:, i])
       



In [1079]:
test_sweep(x,y)